<a href="https://colab.research.google.com/github/joaohppenha/govspendertest/blob/main/Auditoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline Final de Auditoria Inteligente: K-Means & Isolation Forest
Este notebook unifica a análise de gastos públicos aplicando aprendizado de máquina não supervisionado para segmentar perfis de consumo, detectar anomalias comportamentais, classificar casos suspeitos, gerar um relatório executivo em PDF e enviá-lo diretamente para a documentação do projeto no GitHub.

**Instalação e Configuração Inicial**

In [6]:
# Instalação das dependências necessárias
!pip install -q pyarrow fastparquet scikit-learn seaborn matplotlib reportlab requests

import io
import os
import base64
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import userdata

from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

print("Ambiente configurado e bibliotecas importadas com sucesso!")

Ambiente configurado e bibliotecas importadas com sucesso!


## 1. Conexão e Ingestão da Base de Dados
Baixamos a base tratada da camada gold diretamente do repositório utilizando o token de autenticação seguro.

In [7]:
# 1. Recuperação segura do token do GitHub via Secrets do Colab
try:
    github_token = userdata.get('Githubtolken').strip()
except Exception:
    github_token = None

headers = {"Authorization": f"token {github_token}"} if github_token else {}

# 2. Download da base analítica tratada
url_gold = "https://github.com/joaohppenha/govspendertest/raw/refs/heads/main/gold/camada_gold.parquet"
response = requests.get(url_gold, headers=headers)

if response.status_code == 200:
    df = pd.read_parquet(io.BytesIO(response.content))
    print(f"Base gold carregada com sucesso! Total de registros: {df.shape[0]}")
else:
    raise Exception(f"Erro ao baixar camada gold. Status code: {response.status_code}")

Base gold carregada com sucesso! Total de registros: 2810


## 2. Célula: Execução dos Modelos e Cruzamento Inteligente


* **K-Means:** Agrupa os CPFs por similaridade de volume de gastos e ticket médio (divididos em 3 perfis).
* **Isolation Forest:** Identifica pontos fora da curva estatística (anomalias).

In [8]:
# A. K-Means (Perfil de Consumo - Clusterização)
X_kmeans = df[['VOLUME_TOTAL_LOG_SCALED', 'TICKET_MEDIO_LOG_SCALED']].values
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_temp'] = kmeans.fit_predict(X_kmeans)

# Padronizando as etiquetas (0 = Pouco, 1 = Moderado, 2 = Muito)
cluster_ordem = df.groupby('cluster_temp')['VOLUME_TOTAL_LOG_SCALED'].mean().sort_values().index
mapeamento = {cluster_ordem[0]: 0, cluster_ordem[1]: 1, cluster_ordem[2]: 2}
df['cluster_kmeans'] = df['cluster_temp'].map(mapeamento)
df.drop(columns=['cluster_temp'], inplace=True)

# B. Isolation Forest (Detecção de Anomalias)
X_if = df[['VOLUME_TOTAL_LOG_SCALED', 'TICKET_MEDIO_LOG_SCALED', 'VOLATILIDADE_STD_LOG_SCALED']].values
iso = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_pred'] = iso.fit_predict(X_if)
df['anomaly_score'] = iso.decision_function(X_if)

# C. Cruzamento e Classificação de Auditoria
def classificar_auditoria(row):
    if row['anomaly_pred'] == -1 and row['cluster_kmeans'] == 2:
        return "CASO SUSPEITO (Crítico)"
    elif row['anomaly_pred'] == -1:
        return "CASO SUSPEITO (Atípico)"
    else:
        return "Regular"

df['status_auditoria'] = df.apply(classificar_auditoria, axis=1)
df['risco_score'] = np.where(df['anomaly_pred'] == -1, (abs(df['anomaly_score']) * (df['cluster_kmeans'] + 1)), 0)
df_ranking = df.sort_values(by='risco_score', ascending=False)

print("Distribuição da Classificação de Auditoria:")
print(df['status_auditoria'].value_counts())

Distribuição da Classificação de Auditoria:
status_auditoria
Regular                    2669
CASO SUSPEITO (Atípico)      79
CASO SUSPEITO (Crítico)      62
Name: count, dtype: int64


## 3. Geração do Relatório Executivo em PDF

In [12]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.pdfgen import canvas

class NumberedCanvas(canvas.Canvas):
    """Canvas customizado para numeração dinâmica de páginas (Página X de Y)."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pages = []

    def showPage(self):
        self.pages.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        num_pages = len(self.pages)
        for page in self.pages:
            self.__dict__.update(page)
            self.draw_footer(num_pages)
            super().showPage()
        super().save()

    def draw_footer(self, total_pages):
        self.saveState()
        self.setFont("Helvetica", 8)
        self.setFillColor(colors.HexColor("#718096"))

        # Linha separadora do rodapé
        self.setStrokeColor(colors.HexColor("#E2E8F0"))
        self.setLineWidth(0.5)
        self.line(30, 35, letter[0] - 30, 35)

        # Textos de rodapé
        self.drawString(30, 22, "GovSpender Analytics — Relatório de Auditoria Automatizada")
        self.drawRightString(letter[0] - 30, 22, f"Página {self._pageNumber} de {total_pages}")
        self.restoreState()


# Configuração do Documento PDF
pdf_filename = "relatorio_auditoria.pdf"
doc = SimpleDocTemplate(
    pdf_filename,
    pagesize=letter,
    rightMargin=30,
    leftMargin=30,
    topMargin=40,
    bottomMargin=45
)
elementos = []

# Estilos tipográficos profissionais
styles = getSampleStyleSheet()

estilo_titulo = ParagraphStyle(
    'TituloPrincipal',
    parent=styles['Heading1'],
    fontSize=18,
    leading=22,
    textColor=colors.HexColor("#1A365D"),
    fontName='Helvetica-Bold',
    spaceAfter=4
)

estilo_subtitulo = ParagraphStyle(
    'SubTitulo',
    parent=styles['Normal'],
    fontSize=10,
    leading=14,
    textColor=colors.HexColor("#4A5568"),
    fontName='Helvetica',
    spaceAfter=12
)

estilo_secao = ParagraphStyle(
    'SecaoTitulo',
    parent=styles['Heading2'],
    fontSize=12,
    leading=16,
    textColor=colors.HexColor("#2B6CB0"),
    fontName='Helvetica-Bold',
    spaceBefore=12,
    spaceAfter=6
)

estilo_celula = ParagraphStyle(
    'CelulaTabela',
    parent=styles['Normal'],
    fontSize=8.5,
    leading=11,
    textColor=colors.HexColor("#2D3748"),
    fontName='Helvetica'
)

estilo_celula_header = ParagraphStyle(
    'CelulaHeader',
    parent=styles['Normal'],
    fontSize=9,
    leading=11,
    textColor=colors.whitesmoke,
    fontName='Helvetica-Bold',
    alignment=1
)

estilo_critico = ParagraphStyle(
    'CriticoEstilo',
    parent=estilo_celula,
    textColor=colors.HexColor("#C53030"),
    fontName='Helvetica-Bold'
)

estilo_atipico = ParagraphStyle(
    'AtipicoEstilo',
    parent=estilo_celula,
    textColor=colors.HexColor("#DD6B20"),
    fontName='Helvetica-Bold'
)

# 1. Cabeçalho do Relatório
elementos.append(Paragraph("Relatório Oficial de Auditoria & Compliance", estilo_titulo))
elementos.append(Paragraph("Listagem completa de casos suspeitos ordenados por prioridade de risco (K-Means & Isolation Forest).", estilo_subtitulo))
elementos.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor("#2B6CB0"), spaceAfter=15))

# 2. Resumo Executivo
total_geral = len(df)
suspeitos_filtrados = df_ranking[df_ranking['status_auditoria'].str.contains("SUSPEITO")].copy()
total_suspeitos = len(suspeitos_filtrados)
total_criticos = len(suspeitos_filtrados[suspeitos_filtrados['status_auditoria'].str.contains("Crítico")])
total_atipicos = len(suspeitos_filtrados[suspeitos_filtrados['status_auditoria'].str.contains("Atípico")])

texto_resumo = f"""
<b>Sumário Executivo:</b> Foram analisados <b>{total_geral:,}</b> registros na base de dados.
O agente identificou um total de <b>{total_suspeitos}</b> ocorrências suspeitas, divididas em
<b>{total_criticos} casos críticos</b> (Prioridade Alta) e <b>{total_atipicos} casos atípicos</b> (Prioridade Moderada).
Abaixo encontra-se a listagem completa de todos os portadores sinalizados.
"""
elementos.append(Paragraph(texto_resumo, ParagraphStyle('TextoResumo', parent=styles['Normal'], fontSize=9.5, leading=14, textColor=colors.HexColor("#2D3748"))))
elementos.append(Spacer(1, 10))

# 3. Tabela Completa com Todos os Suspeitos Ordenados por Prioridade (Score de Risco)
elementos.append(Paragraph("Listagem Completa de Casos Suspeitos (Ordenados por Prioridade)", estilo_secao))

dados_tabela = [[
    Paragraph("Hash CPF Portador", estilo_celula_header),
    Paragraph("Cluster", estilo_celula_header),
    Paragraph("Status de Auditoria", estilo_celula_header),
    Paragraph("Score de Risco", estilo_celula_header)
]]

# Ordena explicitamente do maior risco para o menor para destacar as prioridades no topo
suspeitos_filtrados = suspeitos_filtrados.sort_values(by='risco_score', ascending=False)

for _, row in suspeitos_filtrados.iterrows():
    status_txt = str(row['status_auditoria'])

    # Aplica estilização condicional baseada na prioridade
    if "Crítico" in status_txt:
        estilo_status = estilo_critico
    else:
        estilo_status = estilo_atipico

    dados_tabela.append([
        Paragraph(str(row['HASH_CPF_PORTADOR']), estilo_celula),
        Paragraph(str(row['cluster_kmeans']), ParagraphStyle('Center', parent=estilo_celula, alignment=1)),
        Paragraph(status_txt, estilo_status),
        Paragraph(f"{row['risco_score']:.2f}", ParagraphStyle('Center', parent=estilo_celula, alignment=1))
    ])

# Criação da tabela expansível por múltiplas páginas
tabela = Table(dados_tabela, colWidths=[212, 60, 180, 100], repeatRows=1)
tabela.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#1A365D")),
    ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
    ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
    ('TOPPADDING', (0, 0), (-1, -1), 5),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 5),
    ('LEFTPADDING', (0, 0), (-1, -1), 8),
    ('RIGHTPADDING', (0, 0), (-1, -1), 8),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.HexColor("#FFFFFF"), colors.HexColor("#F7FAFC")]),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor("#E2E8F0")),
]))

elementos.append(tabela)

# Constrói o PDF permitindo quebra automática em quantas páginas forem necessárias
doc.build(elementos, canvasmaker=NumberedCanvas)
print(f"Relatório PDF completo gerado com sucesso: {pdf_filename} ({total_suspeitos} casos listados).")

Relatório PDF completo gerado com sucesso: relatorio_auditoria.pdf (141 casos listados).


## 4. Envio Automático para o GitHub (Pasta Documentação)

In [13]:
owner = "joaohppenha"
repo = "govspendertest"
branch = "main"
caminho_pasta_repo = "Documentação"
nome_arquivo = "relatorio_auditoria.pdf"
caminho_no_repo = f"{caminho_pasta_repo}/{nome_arquivo}"

with open(pdf_filename, "rb") as f:
    conteudo_binario = f.read()

conteudo_base64 = base64.b64encode(conteudo_binario).decode("utf-8")

url_api = f"https://api.github.com/repos/{owner}/{repo}/contents/{caminho_no_repo}"
api_headers = {
    "Authorization": f"token {github_token}",
    "Accept": "application/vnd.github.v3+json"
}

response_get = requests.get(url_api, headers=api_headers)
sha_arquivo = response_get.json().get("sha") if response_get.status_code == 200 else None

payload = {
    "message": "feat(docs): atualiza relatório automatizado de auditoria de casos suspeitos",
    "content": conteudo_base64,
    "branch": branch
}

if sha_arquivo:
    payload["sha"] = sha_arquivo

response_put = requests.put(url_api, headers=api_headers, json=payload)

if response_put.status_code in [201, 200]:
    print(f"\n[Sucesso] Relatório publicado no GitHub com sucesso!")
    print(f"Acesse em: https://github.com/{owner}/{repo}/tree/{branch}/{caminho_pasta_repo}")
else:
    print(f"\n[Erro] Falha ao enviar para o GitHub. Código: {response_put.status_code}")
    print(response_put.json())


[Sucesso] Relatório publicado no GitHub com sucesso!
Acesse em: https://github.com/joaohppenha/govspendertest/tree/main/Documentação
